# Ferramenta de navegador com visualização ao vivo usando Browser-Use SDK

## Visão Geral

Neste tutorial, aprenderemos como usar o Browser-Use para interagir com a ferramenta Browser do Amazon Bedrock Agentcore e visualizar o navegador ao vivo.


### Detalhes do Tutorial


| Informação          | Detalhes                                                                          |
|:--------------------|:----------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                    |
| Tipo de agente      | Único                                                                             |
| Framework agêntico  | Browser-Use                                                                       |
| Modelo LLM          | Anthropic Claude 3.7 Sonnet                                                       |
| Componentes         | Usando Browser-Use para interagir com a ferramenta de navegador e visualizá-la ao vivo |
| Vertical do tutorial| Vertical                                                                          |
| Complexidade        | Fácil                                                                             |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK, Browser-Use                                   |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como usar o Browser-Use com a ferramenta de navegador e visualizá-lo ao vivo.  

Em nosso exemplo, enviaremos instruções em linguagem natural para o agente Browse-Use executar tarefas no navegador Bedrock Agentcore e visualizar o navegador ao vivo.

<div style="text-align:left">
    <img src="images/browser-tool.png" width="50%"/>
</div>

### Principais Recursos do Tutorial

* Usando a ferramenta de navegador e visualizando-a ao vivo
* Usando Browser-Use com a ferramenta de navegador

## Pré-requisitos

### Para executar este tutorial, você precisará de:
* Python 3.11+
* Credenciais AWS. Sua função/usuário IAM deve ter estas permissões https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Browser-Use SDK

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

### Execute o script de patch abaixo para corrigir o problema do browser-use

In [ ]:
%%writefile patch_browser_use.py

#!/usr/bin/env python3
"""Detectar automaticamente e aplicar patch no session.py do browser_use"""

import os
import shutil
import sys
from pathlib import Path

def find_browser_use_path():
    """Encontrar automaticamente o caminho de instalação do browser_use"""
    try:
        import browser_use
        browser_use_path = Path(browser_use.__file__).parent
        session_file = browser_use_path / "browser" / "session.py"
        return str(session_file)
    except ImportError:
        print("❌ browser_use não instalado. Instale com: pip install browser-use")
        return None

def patch_browser_use():
    # Detectar automaticamente o caminho do arquivo
    file_path = find_browser_use_path()
    if not file_path:
        return False
    
    if not os.path.exists(file_path):
        print(f"❌ Arquivo não encontrado: {file_path}")
        return False
    
    print(f"📁 browser_use encontrado em: {file_path}")
    
    # Criar backup
    backup_path = file_path + ".backup"
    if not os.path.exists(backup_path):
        shutil.copy2(file_path, backup_path)
        print(f"💾 Backup criado: {backup_path}")
    else:
        print(f"📋 Backup já existe: {backup_path}")
    
    # Ler arquivo
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Substituição 1: Adicionar verificação de headers após verificação de cdp_url
    old1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True"
    new1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True\n\n\t\tif headers:\n\t\t\tprofile_kwargs['headers'] = headers"
    
    if old1 in content and "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" not in content:
        content = content.replace(old1, new1)
        print("✅ Verificação de headers adicionada")
    elif "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" in content:
        print("✅ Verificação de headers já existe")
    else:
        print("⚠️ Padrão de verificação de headers não encontrado")
    
    # Substituição 2: Adicionar headers ao CDPClient
    old2 = "self._cdp_client_root = CDPClient(self.cdp_url)"
    new2 = "self._cdp_client_root = CDPClient(self.cdp_url,  additional_headers=self.browser_profile.headers)"
    
    if old2 in content:
        content = content.replace(old2, new2)
        print("✅ Headers adicionados ao CDPClient")
    elif "additional_headers=self.browser_profile.headers" in content:
        print("✅ Headers do CDPClient já existem")
    else:
        print("⚠️ Padrão CDPClient não encontrado")
    
    # Escrever de volta
    with open(file_path, 'w') as f:
        f.write(content)
    
    print("🎉 Patch concluído!")
    return True

if __name__ == "__main__":
    success = patch_browser_use()
    sys.exit(0 if success else 1)

In [ ]:
# Executar o script Python para aplicar o patch para browser-use
!python patch_browser_use.py

In [ ]:
# Reiniciar o Kernel 
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Usando Browser-Use com a ferramenta Browser do Bedrock Agentcore com visualização ao vivo

Aqui, usaremos uma classe auxiliar `BrowserViewerServer` para conectar via Amazon DCV SDK à ferramenta de navegador do Bedrock Agentcore.




In [ ]:
%%writefile live_view_with_browser_use.py
from browser_use import Agent
# from browser_use.browser.session import BrowserSession
from browser_use import Browser, BrowserProfile
from bedrock_agentcore.tools.browser_client import BrowserClient
# from browser_use.browser import BrowserProfile
# from langchain_aws import ChatBedrockConverse
from browser_use.llm import ChatAnthropicBedrock, ChatAWSBedrock
from rich.console import Console
from rich.panel import Panel
from contextlib import suppress
import argparse
import sys
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer
import asyncio
from boto3.session import Session

console = Console()

boto_session = Session()
region = boto_session.region_name


async def run_browser_task(
    browser_session: Browser, bedrock_chat: ChatAnthropicBedrock, task: str
) -> None:
    """
    Executar uma tarefa de automação de navegador usando browser_use

    Args:
        browser_session: Sessão de navegador existente para reutilizar
        bedrock_chat: Instância do modelo de chat Bedrock
        task: Tarefa em linguagem natural para o agente
    """
    try:
        # Mostrar execução da tarefa
        console.print(f"\n[bold blue]🤖 Executando tarefa:[/bold blue] {task}")

        # Criar e executar o agente
        agent = Agent(task=task, llm=bedrock_chat, browser_session=browser_session)

        # Executar com indicador de progresso
        with console.status(
            "[bold green]Executando automação do navegador...[/bold green]", spinner="dots"
        ):
            await agent.run()

        console.print("[bold green]✅ Tarefa concluída com sucesso![/bold green]")

    except Exception as e:
        console.print(f"[bold red]❌ Erro durante execução da tarefa:[/bold red] {str(e)}")
        import traceback

        if console.is_terminal:
            traceback.print_exc()


async def live_view_with_browser_use(prompt, region="us-west-2"):
    """
    Função principal que demonstra visualização de navegador ao vivo com automação do Agent.

    Fluxo de trabalho:
    1. Cria cliente de navegador Amazon Bedrock AgentCore na região us-west-2
    2. Aguarda inicialização do navegador (atraso obrigatório de 10 segundos)
    3. Inicia servidor visualizador ao vivo baseado em DCV na porta 8000 com controle do navegador
    4. Configura múltiplas opções de tamanho de tela (720p a 1440p)
    5. Estabelece sessão de navegador para automação do agente IA via CDP WebSocket
    6. Executa tarefas conduzidas por IA usando o modelo Claude 3.7 Sonnet
    7. Fecha adequadamente todas as sessões e para o cliente do navegador

    Recursos:
    - Visualização de navegador em tempo real através de interface web
    - Funcionalidade manual de assumir/liberar controle
    - Automação IA com biblioteca browser-use
    - Layouts de tela e tamanhos configuráveis
    """
    console.print(
        Panel(
            "[bold cyan]Visualizador de Navegador ao Vivo[/bold cyan]\n\n"
            "Isso demonstra:\n"
            "• Visualização de navegador ao vivo com DCV\n"
            "• Tamanhos de tela configuráveis (não limitado a 900×800)\n"
            "• Callbacks de layout de tela adequados\n\n"
            "[yellow]Nota: Requer arquivos SDK do Amazon DCV[/yellow]",
            title="Visualizador de Navegador ao Vivo",
            border_style="blue",
        )
    )

    try:
        # Passo 1: Criar sessão de navegador
        client = BrowserClient(region)
        client.start()
        
        ws_url, headers = client.generate_ws_headers()

        # Passo 2: Iniciar servidor visualizador
        console.print("\n[cyan]Passo 3: Iniciando servidor visualizador...[/cyan]")
        viewer = BrowserViewerServer(client, port=8000)
        viewer_url = viewer.start(open_browser=True)

        # Passo 3: Mostrar recursos
        console.print("\n[bold green]Recursos do Visualizador:[/bold green]")
        console.print(
            "• Tela padrão: 1600×900 (configurado via callback displayLayout)"
        )
        console.print("• Opções de tamanho: 720p, 900p, 1080p, 1440p")
        console.print("• Atualizações de tela em tempo real")
        console.print("• Funcionalidade de assumir/liberar controle")

        console.print("\n[yellow]Pressione Ctrl+C para parar[/yellow]")

        # Passo 4: Usar browser-use para interagir com o navegador
        # Criar sessão de navegador e modelo persistentes
        browser_session = None
        bedrock_chat = None

        try:
            # Criar perfil de navegador com headers
            browser_profile = BrowserProfile(
                headers=headers,
                timeout=1500000,  # 150 segundos de timeout
            )

            # Criar uma sessão de navegador com CDP URL e keep_alive=True para persistência
            browser_session = Browser(
                cdp_url=ws_url,
                browser_profile=browser_profile,
                keep_alive=True,  # Manter navegador ativo entre tarefas
            )

            # Inicializar a sessão do navegador
            console.print("[cyan]🔄 Inicializando sessão do navegador...[/cyan]")
            await browser_session.start()

            # Criar ChatBedrockConverse uma vez
            bedrock_chat = ChatAnthropicBedrock(
                model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
                aws_region=region,
            )

            console.print(
                "[green]✅ Sessão do navegador inicializada e pronta para tarefas[/green]\n"
            )

            task = prompt

            await run_browser_task(browser_session, bedrock_chat, task)

        finally:
            # Fechar a sessão do navegador
            if browser_session:
                console.print("\n[yellow]🔌 Fechando sessão do navegador...[/yellow]")
                with suppress(Exception):
                    await browser_session.close()
                console.print("[green]✅ Sessão do navegador fechada[/green]")
   
    except Exception as e:
        console.print(f"\n[red]Erro: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Encerrando...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Sessão de navegador terminada")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Instrução de busca no navegador")
    parser.add_argument("--region", default="us-west-2", help="Região AWS")
    args = parser.parse_args()

    asyncio.run(live_view_with_browser_use(
        args.prompt, args.region
    ))

#### Executando o script
Execute o script abaixo. Após o script ser executado (levará algum tempo dependendo da complexidade do prompt), role pela saída para ver o resultado das ações tomadas.

In [ ]:
!python live_view_with_browser_use.py --prompt "Procure por macbooks em amazon.com e extraia os detalhes do primeiro"

## O que aconteceu nos bastidores? 
* Você instanciou um cliente Browser e iniciou uma sessão
* Em seguida, você usou o `BrowserViewerServer` para conectar à sessão do navegador e visualizar a sessão localmente
* Depois, você criou um Agente Browser-Use e passou os detalhes da sessão do navegador para ele
* Você então enviou instruções em linguagem natural para o agente Browser-Use e viu as ações ao vivo


# Parabéns!